# English Data

## Setup Notebook

In [42]:
#import packages

from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize
from sklearn.cluster import AgglomerativeClustering
import sys
from sklearn.model_selection import train_test_split
import hashlib
import json

In [43]:
PROJECT_ROOT = Path.cwd()
ENGLISH_DATA = PROJECT_ROOT / "data" / "raw" / "synthetic-english"
SENTENCES_DIR = ENGLISH_DATA / "generated_sentences"
METADATA_DIR = ENGLISH_DATA / "metadata"

assert SENTENCES_DIR.is_dir(), f"Sentence folder not found: {SENTENCES_DIR}"
assert METADATA_DIR.is_dir(), f"Metadata folder not found: {METADATA_DIR}"

print("Project:", PROJECT_ROOT)
print("English data:", ENGLISH_DATA)

Project: /Users/kmonroygill/Desktop/semantic-change-over-time
English data: /Users/kmonroygill/Desktop/semantic-change-over-time/data/raw/synthetic-english


## Single Word

### Inspect

In [44]:
word_file = SENTENCES_DIR / "2_senses" / "Algol.jsonl"
english_word = pd.read_json(word_file, lines=True)

print("File:", word_file.name)
print("Number of sentences:", len(english_word))
print("Number of gold senses:", english_word["sense_id"].nunique())
print("Years:", sorted(english_word["year"].unique()))

english_word.head()

File: Algol.jsonl
Number of sentences: 300
Number of gold senses: 2
Years: [np.int64(1810), np.int64(1830), np.int64(1850), np.int64(1870), np.int64(1890), np.int64(1910), np.int64(1930), np.int64(1950), np.int64(1970), np.int64(1990)]


,lemma,sense_id,start,end,pos,year,text
0,Algol,algol_nn01-7261115,13,18,NN,1810,That star of algol Which flickers thro' her fl...
1,Algol,algol_nn01-7261115,112,117,NN,1810,"Perseus is a constellation on the ecliptic, be..."
2,Algol,algol_nn01-7261115,74,79,NN,1810,They were like some lone far-flying stars That...
3,Algol,algol_nn01-7261115,21,26,NN,1810,Perseus the Hero and Algol.
4,Algol,algol_nn01-7261115,46,51,NN,1810,A new star was seen between Caput Medusae and ...


In [45]:
sense_counts = pd.crosstab(
    english_word["year"],
    english_word["sense_id"]
)

sense_counts

sense_id,algol_nn01-7261115,algol_nn02-7261822
year,,
1810,30,0
1830,30,0
1850,30,0
1870,30,0
1890,30,0
1910,30,0
1930,30,0
1950,30,0
1970,21,9


In [46]:
first = english_word.iloc[0]
text = first["text"]
s = first["start"]
e = first["end"]

print(text) #full sentence
print(text[s:e]) #just word

That star of algol Which flickers thro' her flight.
algol


### Prepare Data

In [47]:
#reformat English data so format matches Swedish data

english_usages = english_word.rename(columns={"lemma": "word"}).copy()

english_usages["period_label"] = english_usages["year"].astype(str)
english_usages["sentence_id"] = [
    f"{word_file.stem}_{i}" for i in english_usages.index
]

english_usages = english_usages.drop(columns=["sense_id"])

english_usages.head()

,word,start,end,pos,year,text,period_label,sentence_id
0,Algol,13,18,NN,1810,That star of algol Which flickers thro' her fl...,1810,Algol_0
1,Algol,112,117,NN,1810,"Perseus is a constellation on the ecliptic, be...",1810,Algol_1
2,Algol,74,79,NN,1810,They were like some lone far-flying stars That...,1810,Algol_2
3,Algol,21,26,NN,1810,Perseus the Hero and Algol.,1810,Algol_3
4,Algol,46,51,NN,1810,A new star was seen between Caput Medusae and ...,1810,Algol_4


In [48]:
english_gold = english_usages[
    ["word", "period_label", "sentence_id"]
].copy()

sense_labels, sense_ids = pd.factorize(
    english_word["sense_id"], sort=True
)

english_gold["label"] = [[int(label)] for label in sense_labels]

english_gold.head()

,word,period_label,sentence_id,label
0,Algol,1810,Algol_0,[0]
1,Algol,1810,Algol_1,[0]
2,Algol,1810,Algol_2,[0]
3,Algol,1810,Algol_3,[0]
4,Algol,1810,Algol_4,[0]


### Generate Embeddings

In [49]:
#load/setup models

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

#roberta
roberta_name = "FacebookAI/xlm-roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name, use_fast=True)
roberta_model = AutoModel.from_pretrained(roberta_name).to(device).eval()

#lexeme
lexeme_name = "pierluigic/xl-lexeme"
lexeme_tokenizer = AutoTokenizer.from_pretrained(lexeme_name, use_fast=True)
lexeme_model = AutoModel.from_pretrained(lexeme_name).to(device).eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11437.39it/s]
[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 31375.63it/s]


In [50]:
def get_roberta_vector(row):
    sentence = row["text"]
    start = int(row["start"])
    end = int(row["end"])

    encoded = roberta_tokenizer(
        sentence,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()

    target_tokens = []
    for i, (token_start, token_end) in enumerate(offsets):
        if token_end > token_start and token_start < end and token_end > start:
            target_tokens.append(i)

    if not target_tokens:
        raise ValueError(f"Target word not found: {row['sentence_id']}")

    encoded = encoded.to(device)

    with torch.no_grad():
        output = roberta_model(**encoded)

    target_vectors = output.last_hidden_state[0, target_tokens]
    vector = target_vectors.mean(dim=0)

    return vector.cpu().numpy()

In [51]:
#test single occurrence

first = english_usages.iloc[0]
vector = get_roberta_vector(first)

print("Target word:", first["text"][first["start"]:first["end"]])
print("Vector shape:", vector.shape)

Target word: algol
Vector shape: (768,)


In [52]:
def get_lexeme_vector(row):
    text = row["text"]
    start = int(row["start"])
    end = int(row["end"])

    marked_text = f"{text[:start]}<t>{text[start:end]}</t>{text[end:]}"
    encoded = lexeme_tokenizer(marked_text, return_tensors="pt")

    if encoded["input_ids"].shape[1] > 512:
        raise ValueError("Sentence needs a shorter context window.")

    encoded = encoded.to(device)

    with torch.no_grad():
        output = lexeme_model(**encoded)

    valid_tokens = encoded["attention_mask"][0].bool()
    token_vectors = output.last_hidden_state[0][valid_tokens]
    vector = token_vectors.mean(dim=0)

    return vector.cpu().numpy()

In [53]:
#generate embedding shapes

roberta_vectors = []
for _, row in tqdm(english_usages.iterrows(), total=len(english_usages)):
    roberta_vectors.append(get_roberta_vector(row))
english_roberta_vectors = np.stack(roberta_vectors)

lexeme_vectors = []
for _, row in tqdm(english_usages.iterrows(), total=len(english_usages)):
    lexeme_vectors.append(get_lexeme_vector(row))
english_lexeme_vectors = np.stack(lexeme_vectors)

print("Roberta embedding shape:", english_roberta_vectors.shape)
print("Lexeme embedding shape:", english_lexeme_vectors.shape)

100%|██████████| 300/300 [00:05<00:00, 52.79it/s]

Roberta embedding shape: (300, 768)
Lexeme embedding shape: (300, 1024)


### Blend and Cluster - Using Swedish Settings

In [54]:
assert (
    len(english_roberta_vectors)
    == len(english_lexeme_vectors)
    == len(english_usages)
)

english_roberta_distances = pairwise_distances(
    normalize(english_roberta_vectors), metric="cosine"
)

english_lexeme_distances = pairwise_distances(
    normalize(english_lexeme_vectors), metric="cosine"
)

np.fill_diagonal(english_roberta_distances, 0)
np.fill_diagonal(english_lexeme_distances, 0)

print("RoBERTa distances:", english_roberta_distances.shape)
print("XL-LEXEME distances:", english_lexeme_distances.shape)

RoBERTa distances: (300, 300)
XL-LEXEME distances: (300, 300)


In [55]:
#load cached Swedish embeddings

cache_folder = (
    PROJECT_ROOT / "outputs" / "embeddings" / "subtask1_embedding_cache" / "SV"
)

# Same tuning words used in the group notebook.
SV_tuning_words = [
    "motiv", "kvinna", "fru", "fröken", "förort", "skär", "suga"
]

with np.load(cache_folder / "xl_lexeme_vectors.npz") as cache:
    swedish_lexeme_vectors = {
        word: cache[word] for word in SV_tuning_words
    }

with np.load(cache_folder / "xlmr_vectors.npz") as cache:
    swedish_roberta_vectors = {
        word: cache[word] for word in SV_tuning_words
    }

print("Loaded Swedish embeddings for:", SV_tuning_words)

Loaded Swedish embeddings for: ['motiv', 'kvinna', 'fru', 'fröken', 'förort', 'skär', 'suga']


In [56]:
lexeme_pairs = []
roberta_pairs = []

for word in SV_tuning_words:
    lexeme_vectors = swedish_lexeme_vectors[word]
    roberta_vectors = swedish_roberta_vectors[word]

    assert len(lexeme_vectors) == len(roberta_vectors)

    lexeme_matrix = pairwise_distances(
        normalize(lexeme_vectors), metric="cosine"
    )
    roberta_matrix = pairwise_distances(
        normalize(roberta_vectors), metric="cosine"
    )

    np.fill_diagonal(lexeme_matrix, 0)
    np.fill_diagonal(roberta_matrix, 0)

    upper_triangle = np.triu_indices(len(lexeme_vectors), k=1)
    lexeme_pairs.append(lexeme_matrix[upper_triangle])
    roberta_pairs.append(roberta_matrix[upper_triangle])

lexeme_typical = np.median(np.concatenate(lexeme_pairs))
roberta_typical = np.median(np.concatenate(roberta_pairs))

if roberta_typical <= 0:
    raise ValueError("Cannot scale distances with a zero median.")

xlmr_distance_scale = lexeme_typical / roberta_typical

print("XLM-RoBERTa distance multiplier:", xlmr_distance_scale)

XLM-RoBERTa distance multiplier: 13.111755


In [57]:
english_blended_distances = (
    0.5 * english_lexeme_distances
    + 0.5 * xlmr_distance_scale * english_roberta_distances
)

english_blended_distances = np.maximum(english_blended_distances, 0)
english_blended_distances = (
    english_blended_distances + english_blended_distances.T
) / 2
np.fill_diagonal(english_blended_distances, 0)

print("Blended distance shape:", english_blended_distances.shape)

Blended distance shape: (300, 300)


In [58]:
english_pipeline_settings = {
    "lexeme": (english_lexeme_distances, 0.4),
    "roberta": (english_roberta_distances, 0.010),
    "blend": (english_blended_distances, 0.3),
}

english_predictions = {}

for approach, (distances, cutoff) in english_pipeline_settings.items():
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=cutoff
    ).fit_predict(distances)

    predictions = english_usages[
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]

    english_predictions[approach] = predictions

    print(f"{approach}: {len(np.unique(labels))} clusters")

lexeme: 2 clusters
roberta: 6 clusters
blend: 2 clusters


### Evaluate Predictions

In [59]:
evaluation_folder = PROJECT_ROOT / "src" / "evaluation"

sys.path.insert(0, str(evaluation_folder / "python-bcubed"))
sys.path.insert(0, str(evaluation_folder / "semeval2027-task3"))

from evaluation import evaluate_subtask1

In [60]:
gold_folder = PROJECT_ROOT / "data" / "processed" / "gold" / "ENG"
prediction_folder = PROJECT_ROOT / "outputs" / "predictions" / "ENG"

gold_folder.mkdir(parents=True, exist_ok=True)
prediction_folder.mkdir(parents=True, exist_ok=True)

gold_path = gold_folder / f"{word_file.stem}_gold.jsonl"
english_gold.to_json(gold_path, orient="records", lines=True)

english_results = []

for approach, predictions in english_predictions.items():
    prediction_path = prediction_folder / f"{word_file.stem}_{approach}.jsonl"
    predictions.to_json(prediction_path, orient="records", lines=True)

    clustering, change = evaluate_subtask1(gold_path, prediction_path)

    english_results.append({
        "approach": approach,
        "1a_F1": clustering["fcubed"],
        "1b_Spearman": change,
    })

english_results = pd.DataFrame(english_results)
english_results

,approach,1a_F1,1b_Spearman
0,lexeme,0.885557,-0.312019
1,roberta,0.892631,-0.163188
2,blend,0.927500,-0.283473


In [61]:
#predicted sense frequencies

for approach, predictions in english_predictions.items():
    print(f"\n{approach}")

    sense_counts = pd.crosstab(
        predictions["period_label"],
        predictions["label"].apply(lambda labels: labels[0])
    )

    display(sense_counts)


lexeme


label,0,1
period_label,,
1810,29,1
1830,29,1
1850,29,1
1870,27,3
1890,29,1
1910,30,0
1930,26,4
1950,28,2
1970,29,1



roberta


label,0,1,2,3,4,5
period_label,,,,,,
1810,27,2,0,0,1,0
1830,29,0,0,1,0,0
1850,28,0,2,0,0,0
1870,29,1,0,0,0,0
1890,29,1,0,0,0,0
1910,29,1,0,0,0,0
1930,28,1,0,0,0,1
1950,28,2,0,0,0,0
1970,30,0,0,0,0,0



blend


label,0,1
period_label,,
1810,30,0
1830,30,0
1850,30,0
1870,30,0
1890,30,0
1910,30,0
1930,29,1
1950,30,0
1970,30,0


In [62]:
#cluster membership vs gold labels

for approach, predictions in english_predictions.items():
    print(f"\n{approach}")

    comparison = english_gold.merge(
        predictions,
        on=["word", "period_label", "sentence_id"],
        suffixes=("_gold", "_predicted"),
        validate="one_to_one"
    )

    display(pd.crosstab(
        comparison["label_gold"].apply(lambda labels: labels[0]),
        comparison["label_predicted"].apply(lambda labels: labels[0])
    ))


lexeme


label_predicted,0,1
label_gold,,
0,266,13
1,18,3



roberta


label_predicted,0,1,2,3,4,5
label_gold,,,,,,
0,266,8,2,1,1,1
1,21,0,0,0,0,0



blend


label_predicted,0,1
label_gold,,
0,278,1
1,21,0


In [63]:
#examples from each sense

sense_examples = english_usages.merge(
    english_gold,
    on=["word", "period_label", "sentence_id"],
    validate="one_to_one"
)

sense_examples["gold_sense"] = sense_examples["label"].apply(
    lambda labels: labels[0]
)

for sense, rows in sense_examples.groupby("gold_sense"):
    print(f"\nGold sense {sense} — {len(rows)} occurrences")

    examples = rows.sample(n=min(4, len(rows)), random_state=42)

    for _, row in examples.iterrows():
        print(f"{row['year']}: {row['text']}")


Gold sense 0 — 279 occurrences
1830: The very stars to the view unknown of old Are here, with Algol and Betelgeuse.
1890: Algol.
1930: In the constellation Perseus (the hero who wrestled with the sea-monster) there are two bright stars. One of these is Algol, and the other Alpheida.
1890: He will find Algol (Perseus) at its lowest.

Gold sense 1 — 21 occurrences
1970: The new Algol will be introduced this term in the Department of Computer Science and Engineering.
1990: The Algol program was written for the IBM PC.
1990: For those with an aptitude to work on problems with mathematical and programming skills, Algol is an ideal language.
1970: A computer program written in Algol is much more concise than the same program in basic and takes less time to run.


## Full English Corpus

### Prepare Data

In [64]:
def prepare_english_word(word_file):
    english_word = pd.read_json(word_file, lines=True)

    assert english_word["sense_id"].notna().all(), (
        f"Missing sense labels: {word_file}"
    )

    english_usages = english_word.rename(
        columns={"lemma": "word"}
    ).copy()

    english_usages["period_label"] = english_usages["year"].astype(str)
    english_usages["sentence_id"] = [
        f"{word_file.stem}_{i}" for i in english_usages.index
    ]

    english_usages = english_usages.drop(columns=["sense_id"])

    english_gold = english_usages[
        ["word", "period_label", "sentence_id"]
    ].copy()

    sense_labels, sense_ids = pd.factorize(
        english_word["sense_id"], sort=True
    )

    english_gold["label"] = [[int(label)] for label in sense_labels]

    return english_usages, english_gold

def prepare_english_context(row, max_tokens=512):
    text = row["text"]
    if not isinstance(text, str) or not text.strip():
        return None, "Missing or empty sentence"

    try:
        start = int(row["start"])
        end = int(row["end"])
        if start != row["start"] or end != row["end"]:
            return None, "Non-integer target offsets"
    except (TypeError, ValueError, OverflowError):
        return None, "Invalid target offsets"

    if not 0 <= start < end <= len(text):
        return None, "Invalid target offsets"
    if not text[start:end].strip():
        return None, "Empty or whitespace-only target"

    # Keep the target intact and reduce surrounding context only when needed.
    radius = max(start, len(text) - end)
    while True:
        left = max(0, start - radius)
        right = min(len(text), end + radius)
        context = text[left:right]
        context_start = start - left
        context_end = end - left

        roberta_encoded = roberta_tokenizer(
            context, return_offsets_mapping=True,
            truncation=False, verbose=False,
        )
        target_found = any(
            token_end > token_start
            and token_start < context_end
            and token_end > context_start
            for token_start, token_end in roberta_encoded["offset_mapping"]
        )
        if not target_found:
            return None, "Target span has no RoBERTa tokens"

        marked_context = (
            f"{context[:context_start]}<t>"
            f"{context[context_start:context_end]}</t>"
            f"{context[context_end:]}"
        )
        lexeme_encoded = lexeme_tokenizer(
            marked_context, truncation=False, verbose=False,
        )

        if (
            len(roberta_encoded["input_ids"]) <= max_tokens
            and len(lexeme_encoded["input_ids"]) <= max_tokens
        ):
            return (context, context_start, context_end), None

        if radius == 0:
            return None, "Target alone exceeds a model token limit"
        radius //= 2


In [65]:
word_files = [
    path
    for path in sorted(SENTENCES_DIR.glob("*_senses/*.jsonl"))
    if path.stat().st_size > 0
]

assert len({path.stem for path in word_files}) == len(word_files)

cache_filenames_by_word = {
    source_file.stem: f"{source_file.stem}.npz"
    for source_file in word_files
}
cache_filenames_by_word["jack"] = "jack1.npz"
cache_filenames_by_word["Jack"] = "Jack2.npz"

assert len({name.casefold() for name in cache_filenames_by_word.values()}) == len(
    cache_filenames_by_word
)

english_data_by_word = {}
english_gold_by_word = {}

for word_file in tqdm(word_files, desc="Preparing English corpus"):
    usages, gold = prepare_english_word(word_file)

    english_data_by_word[word_file.stem] = usages
    english_gold_by_word[word_file.stem] = gold

print("Words prepared:", len(english_data_by_word))
print(
    "Total occurrences:",
    sum(len(rows) for rows in english_data_by_word.values())
)

Preparing English corpus: 100%|██████████| 2344/2344 [00:08<00:00, 271.68it/s]

Words prepared: 2344
Total occurrences: 672120


In [66]:
excluded_parts = []
cropped_parts = []
empty_words = []

for word, usages in tqdm(
    english_data_by_word.items(),
    total=len(english_data_by_word),
    desc="Preparing target contexts"
):
    gold = english_gold_by_word[word]
    assert usages["sentence_id"].tolist() == gold["sentence_id"].tolist(), word

    prepared_usages = usages.copy()
    valid_indices = []

    for index, row in usages.iterrows():
        prepared_context, reason = prepare_english_context(row)

        if prepared_context is None:
            excluded_parts.append({**row.to_dict(), "reason": reason})
            continue

        context, start, end = prepared_context
        if context != row["text"]:
            cropped_parts.append({
                "word": word,
                "sentence_id": row["sentence_id"],
                "original_text": row["text"],
                "text": context,
                "original_start": row["start"],
                "start": start,
                "original_end": row["end"],
                "end": end,
            })

        prepared_usages.at[index, "text"] = context
        prepared_usages.at[index, "start"] = start
        prepared_usages.at[index, "end"] = end
        valid_indices.append(index)

    # Preserve original sentence IDs and remove the same rows from gold.
    english_data_by_word[word] = prepared_usages.loc[valid_indices].copy()
    english_gold_by_word[word] = gold.loc[valid_indices].copy()

    if not valid_indices:
        empty_words.append(word)

excluded_occurrences = pd.DataFrame(
    excluded_parts,
    columns=["word", "sentence_id", "text", "start", "end", "reason"],
)
cropped_occurrences = pd.DataFrame(
    cropped_parts,
    columns=[
        "word", "sentence_id", "original_text", "text",
        "original_start", "start", "original_end", "end",
    ],
)

exclusion_folder = (
    PROJECT_ROOT / "data" / "processed" / "subsets" / "ENG"
)
exclusion_folder.mkdir(parents=True, exist_ok=True)
excluded_occurrences.to_csv(
    exclusion_folder / "excluded_occurrences.csv", index=False
)
cropped_occurrences.to_csv(
    exclusion_folder / "cropped_occurrences.csv", index=False
)

print("Excluded occurrences:", len(excluded_occurrences))
print("Cropped occurrences:", len(cropped_occurrences))
print("Words with no valid occurrences:", empty_words)

assert not empty_words, "Resolve these words before continuing."

Preparing target contexts: 100%|██████████| 2344/2344 [01:48<00:00, 21.53it/s]

Excluded occurrences: 468
Cropped occurrences: 10
Words with no valid occurrences: []


In [67]:
english_corpus_gold = pd.concat(
    english_gold_by_word.values(),
    ignore_index=True
)

corpus_gold_folder = (
    PROJECT_ROOT / "data" / "processed" / "gold" / "ENG"
)
corpus_gold_folder.mkdir(parents=True, exist_ok=True)

corpus_gold_path = corpus_gold_folder / "subtask1.jsonl"
english_corpus_gold.to_json(
    corpus_gold_path, orient="records", lines=True
)

print("Saved gold labels:", corpus_gold_path)
print("Occurrences:", len(english_corpus_gold))

Saved gold labels: /Users/kmonroygill/Desktop/semantic-change-over-time/data/processed/gold/ENG/subtask1.jsonl
Occurrences: 671652


### Generate English Embeddings

In [68]:
def get_english_word_vectors(usages):
    roberta_vectors = []
    for _, row in tqdm(
        usages.iterrows(), total=len(usages), desc="RoBERTa"
    ):
        roberta_vectors.append(get_roberta_vector(row))

    lexeme_vectors = []
    for _, row in tqdm(
        usages.iterrows(), total=len(usages), desc="XL-LEXEME"
    ):
        lexeme_vectors.append(get_lexeme_vector(row))

    return np.stack(roberta_vectors), np.stack(lexeme_vectors)

In [69]:
def get_english_cache_signature(usages):
    signature_data = {
        "inputs": usages.to_json(orient="records"),
        "roberta": roberta_name,
        "roberta_revision": getattr(roberta_model.config, "_commit_hash", None),
        "lexeme": lexeme_name,
        "lexeme_revision": getattr(lexeme_model.config, "_commit_hash", None),
        "extraction_version": 1,
    }
    return hashlib.sha256(
        json.dumps(signature_data, sort_keys=True).encode()
    ).hexdigest()

embedding_folder = PROJECT_ROOT / "outputs" / "embeddings" / "subtask1_embedding_cache" / "ENG"
embedding_folder.mkdir(parents=True, exist_ok=True)

failure_path = embedding_folder / "failures.jsonl"

completed = 0
cached = 0
failed = 0

for word, usages in tqdm(
    english_data_by_word.items(),
    total=len(english_data_by_word),
    desc="English corpus"
):
    cache_path = embedding_folder / cache_filenames_by_word[word]

    signature = get_english_cache_signature(usages)

    try:
        if cache_path.exists():
            with np.load(cache_path, allow_pickle=False) as saved:
                if (
                    saved["signature"].item() == signature
                    and np.array_equal(
                        saved["sentence_ids"],
                        usages["sentence_id"].to_numpy(dtype=str)
                    )
                    and saved["roberta"].shape == (len(usages), 768)
                    and saved["lexeme"].shape == (len(usages), 1024)
                ):
                    cached += 1
                    continue

        roberta_vectors, lexeme_vectors = get_english_word_vectors(usages)

        assert roberta_vectors.shape == (len(usages), 768)
        assert lexeme_vectors.shape == (len(usages), 1024)
        assert np.isfinite(roberta_vectors).all()
        assert np.isfinite(lexeme_vectors).all()

        # Publish the cache only after the complete file is written.
        temporary_path = cache_path.with_suffix(".tmp")
        with temporary_path.open("wb") as file:
            np.savez_compressed(
                file,
                roberta=roberta_vectors,
                lexeme=lexeme_vectors,
                sentence_ids=usages["sentence_id"].to_numpy(dtype=str),
                signature=np.array(signature),
            )
        temporary_path.replace(cache_path)
        completed += 1

    except Exception as error:
        failed += 1
        with failure_path.open("a") as file:
            file.write(json.dumps({
                "word": word,
                "error": repr(error),
            }) + "\n")
        tqdm.write(f"Failed: {word}: {error}")

print(f"Newly saved: {completed}; already cached: {cached}; failed: {failed}")

English corpus: 100%|██████████| 2344/2344 [00:24<00:00, 96.19it/s] 

Newly saved: 0; already cached: 2344; failed: 0


In [70]:
missing_words = []

for word, usages in tqdm(
    english_data_by_word.items(),
    total=len(english_data_by_word),
    desc="Checking embedding caches"
):
    cache_path = embedding_folder / cache_filenames_by_word[word]
    try:
        with np.load(cache_path, allow_pickle=False) as saved:
            valid_cache = (
                saved["signature"].item() == get_english_cache_signature(usages)
                and np.array_equal(
                    saved["sentence_ids"],
                    usages["sentence_id"].to_numpy(dtype=str)
                )
                and saved["roberta"].shape == (len(usages), 768)
                and saved["lexeme"].shape == (len(usages), 1024)
                and np.isfinite(saved["roberta"]).all()
                and np.isfinite(saved["lexeme"]).all()
            )
    except Exception:
        valid_cache = False

    if not valid_cache:
        missing_words.append(word)

print("Missing or invalid caches:", len(missing_words))
print(missing_words)

assert not missing_words, (
    "Resolve failed words and rerun embedding generation before loading caches."
)

Checking embedding caches: 100%|██████████| 2344/2344 [00:43<00:00, 53.34it/s]

Missing or invalid caches: 0
[]


### Load Cache & Cluster Functions

In [71]:
def load_english_word_vectors(word, usages):
    cache_path = embedding_folder / cache_filenames_by_word[word]

    with np.load(cache_path, allow_pickle=False) as saved:
        sentence_ids = saved["sentence_ids"]
        roberta_vectors = saved["roberta"]
        lexeme_vectors = saved["lexeme"]
        assert saved["signature"].item() == get_english_cache_signature(usages), (
            f"Cached inputs or model configuration do not match: {word}"
        )

    assert np.array_equal(
        sentence_ids,
        usages["sentence_id"].to_numpy(dtype=str)
    ), f"Cached sentence IDs do not match: {word}"

    assert roberta_vectors.shape == (len(usages), 768), word
    assert lexeme_vectors.shape == (len(usages), 1024), word
    assert np.isfinite(roberta_vectors).all(), word
    assert np.isfinite(lexeme_vectors).all(), word

    return roberta_vectors, lexeme_vectors

In [72]:
english_roberta_vectors_by_word = {}
english_lexeme_vectors_by_word = {}

for word, usages in tqdm(
    english_data_by_word.items(),
    total=len(english_data_by_word),
    desc="Loading cached embeddings"
):
    roberta_vectors, lexeme_vectors = load_english_word_vectors(word, usages)

    english_roberta_vectors_by_word[word] = roberta_vectors
    english_lexeme_vectors_by_word[word] = lexeme_vectors

Loading cached embeddings: 100%|██████████| 2344/2344 [00:24<00:00, 95.34it/s] 


In [73]:
def get_word_distances(roberta_vectors, lexeme_vectors):
    assert len(roberta_vectors) == len(lexeme_vectors)

    roberta_distances = pairwise_distances(
        normalize(roberta_vectors), metric="cosine"
    )

    lexeme_distances = pairwise_distances(
        normalize(lexeme_vectors), metric="cosine"
    )

    np.fill_diagonal(roberta_distances, 0)
    np.fill_diagonal(lexeme_distances, 0)

    return {
        "roberta": roberta_distances,
        "lexeme": lexeme_distances,
    }

In [93]:
def cluster_word(usages, word_distances, settings):
    blend_settings = settings["blend"]
    lexeme_weight = blend_settings["lexeme_weight"]

    blended_distances = (
        lexeme_weight * word_distances["lexeme"]
        + (1 - lexeme_weight)
        * blend_settings["distance_scale"]
        * word_distances["roberta"]
    )

    pipeline_distances = {
        "lexeme": word_distances["lexeme"],
        "roberta": word_distances["roberta"],
        "blend": blended_distances,
    }

    predictions_by_approach = {}

    for approach, distances in pipeline_distances.items():
        distances = np.maximum(distances, 0)
        distances = (distances + distances.T) / 2
        np.fill_diagonal(distances, 0)

        labels = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=settings[approach]["distance_cutoff"],
        ).fit_predict(distances)

        predictions = usages[
            ["word", "period_label", "sentence_id"]
        ].copy()

        predictions["label"] = [[int(label)] for label in labels]
        predictions_by_approach[approach] = predictions

    return predictions_by_approach

### 1. English - Using Swedish Settings

In [75]:
swedish_settings = {
    "lexeme": {"distance_cutoff": 0.4},
    "roberta": {"distance_cutoff": 0.010},
    "blend": {
        "distance_cutoff": 0.3,
        "lexeme_weight": 0.5,
        "distance_scale": float(xlmr_distance_scale),
    },
}

#### Blend and Cluster

In [76]:
assert set(english_data_by_word) == set(english_roberta_vectors_by_word)
assert set(english_data_by_word) == set(english_lexeme_vectors_by_word)

english_prediction_parts = {
    approach: [] for approach in swedish_settings
}

for word, usages in tqdm(
    english_data_by_word.items(),
    total=len(english_data_by_word),
    desc="Clustering English corpus"
):
    word_distances = get_word_distances(
        english_roberta_vectors_by_word[word],
        english_lexeme_vectors_by_word[word],
    )

    word_predictions = cluster_word(
        usages,
        word_distances,
        swedish_settings,
    )

    for approach, predictions in word_predictions.items():
        english_prediction_parts[approach].append(predictions)

english_SV_settings_predictions = {
    approach: pd.concat(parts, ignore_index=True)
    for approach, parts in english_prediction_parts.items()
}

Clustering English corpus: 100%|██████████| 2344/2344 [00:19<00:00, 122.81it/s]


In [77]:
english_baseline = english_corpus_gold[
    ["word", "period_label", "sentence_id"]
].copy()

english_baseline["label"] = [[0] for _ in range(len(english_baseline))]

english_SV_settings_predictions["Single-cluster baseline"] = english_baseline

In [78]:
english_SV_settings_prediction_folder = (
    PROJECT_ROOT / "outputs" / "predictions" / "ENG" / "swedish_settings"
)
english_SV_settings_prediction_folder.mkdir(parents=True, exist_ok=True)

for approach, predictions in english_SV_settings_predictions.items():
    predictions.to_json(
        english_SV_settings_prediction_folder / f"{approach}.jsonl",
        orient="records",
        lines=True,
    )

    print(f"{approach}: {len(predictions):,} predictions saved")

lexeme: 671,652 predictions saved
roberta: 671,652 predictions saved
blend: 671,652 predictions saved
Single-cluster baseline: 671,652 predictions saved


#### Evaluate

In [79]:
from evaluation import (
    load_subtask1_labels, compute_bcubed, calculate_JSD_distances,
    filter_jsd_by_min_instances, compute_spearman,
)

english_SV_settings_results = []
match_columns = ["word", "period_label", "sentence_id"]

for approach, predictions in english_SV_settings_predictions.items():
    coverage = english_corpus_gold[match_columns].merge(
        predictions[match_columns],
        on=match_columns,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    assert coverage["_merge"].eq("both").all(), approach

    prediction_path = english_SV_settings_prediction_folder / f"{approach}.jsonl"
    predicted_counts, gold_counts, predicted_labels, gold_labels = (
        load_subtask1_labels(prediction_path, corpus_gold_path)
    )

    # All words contribute to clustering F1.
    clustering = compute_bcubed(gold_labels, predicted_labels, prediction_path)

    # Temporal change requires at least two periods.
    temporal_gold_counts = {
        word: periods for word, periods in gold_counts.items()
        if len(periods) >= 2
    }
    temporal_predicted_counts = {
        word: predicted_counts[word] for word in temporal_gold_counts
    }
    print(approach, "words excluded from 1b only:",
          sorted(set(gold_counts) - set(temporal_gold_counts)))

    gold_jsd = calculate_JSD_distances(temporal_gold_counts)
    predicted_jsd = calculate_JSD_distances(temporal_predicted_counts)
    gold_jsd = filter_jsd_by_min_instances(
        gold_jsd, temporal_gold_counts, min_instances=10,
    )
    change = compute_spearman(gold_jsd, predicted_jsd, prediction_path)["rho"]

    english_SV_settings_results.append({
        "approach": approach,
        "settings": "Swedish",
        "words": english_corpus_gold["word"].nunique(),
        "occurrences": len(english_corpus_gold),
        "1a_F1": clustering["fcubed"],
        "1b_Spearman": change,
    })

english_SV_settings_results = pd.DataFrame(english_SV_settings_results)
english_SV_settings_results

lexeme words excluded from 1b only: ['downsize', 'fecker']
roberta words excluded from 1b only: ['downsize', 'fecker']
blend words excluded from 1b only: ['downsize', 'fecker']
Single-cluster baseline words excluded from 1b only: ['downsize', 'fecker']


/Users/kmonroygill/Desktop/semantic-change-over-time/src/evaluation/semeval2027-task3/evaluation.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pvalue = spearmanr(gold_scores, predicted_scores)


,approach,settings,words,occurrences,1a_F1,1b_Spearman
0,lexeme,Swedish,2344,671652,0.643632,0.326136
1,roberta,Swedish,2344,671652,0.557112,0.198109
2,blend,Swedish,2344,671652,0.640946,0.314427
3,Single-cluster baseline,Swedish,2344,671652,0.582897,0.000000


In [119]:
evaluation_output_folder = (
    PROJECT_ROOT / "outputs" / "evaluations" / "ENG"
)
evaluation_output_folder.mkdir(parents=True, exist_ok=True)

english_SV_settings_results.to_csv(
    evaluation_output_folder / "subtask1_english_with_SV_settings.csv",
    index=False,
)

english_SV_settings_results = pd.DataFrame(english_SV_settings_results)
english_SV_settings_results

,approach,settings,words,occurrences,1a_F1,1b_Spearman
0,lexeme,Swedish,2344,671652,0.643632,0.326136
1,roberta,Swedish,2344,671652,0.557112,0.198109
2,blend,Swedish,2344,671652,0.640946,0.314427
3,Single-cluster baseline,Swedish,2344,671652,0.582897,0.000000


### 2. English Fine-Tuning

#### Prepare and Split Data

In [81]:
english_inventory = pd.DataFrame([
    {
        "file_path": str(path.relative_to(SENTENCES_DIR)),
        "word": path.stem,
        "sense_count": int(path.parent.name.split("_")[0]),
    }
    for path in sorted(SENTENCES_DIR.glob("*_senses/*.jsonl"))
    if path.stat().st_size > 0
])

print("Total word files:", len(english_inventory))

folder_counts = (
    english_inventory.groupby("sense_count")
    .size()
    .rename("word_count")
)

display(folder_counts)

Total word files: 2344


sense_count
2     707
3     442
4     315
5     215
6     153
7     102
8      82
9      50
10     63
11     40
12     26
13     30
14     20
15     13
16     10
17     11
18      9
19      1
20     11
21      8
22      4
24      7
25      8
26      4
27      4
28      1
29      3
35      4
36      1
Name: word_count, dtype: int64

In [82]:
assert english_inventory["word"].is_unique

split_strata = english_inventory["sense_count"].clip(upper=18)

english_tuning, english_heldout = train_test_split(
    english_inventory,
    test_size=0.30,
    random_state=42,
    stratify=split_strata,
)

english_tuning = english_tuning.copy()
english_heldout = english_heldout.copy()

english_tuning["split"] = "tuning"
english_heldout["split"] = "heldout"

english_split = pd.concat(
    [english_tuning, english_heldout],
    ignore_index=True
)

display(pd.crosstab(
    english_split["sense_count"],
    english_split["split"]
))

split,heldout,tuning
sense_count,,
2,212,495
3,133,309
4,95,220
5,64,151
6,46,107
7,31,71
8,25,57
9,15,35
10,19,44


In [83]:
split_folder = PROJECT_ROOT / "data" / "processed" / "subsets" / "ENG"
split_folder.mkdir(parents=True, exist_ok=True)

english_split.to_csv(
    split_folder / "word_split.csv",
    index=False
)

english_tuning_words = english_tuning["word"].tolist()
english_heldout_words = english_heldout["word"].tolist()

assert set(english_tuning_words).isdisjoint(english_heldout_words)
assert len(english_split) == len(english_inventory)

print("Saved:", split_folder / "word_split.csv")

english_heldout_prediction_folder = (
    PROJECT_ROOT / "outputs" / "predictions" / "ENG"
)
english_heldout_prediction_folder.mkdir(parents=True, exist_ok=True)

Saved: /Users/kmonroygill/Desktop/semantic-change-over-time/data/processed/subsets/ENG/word_split.csv


In [84]:
english_tuning_gold = pd.concat(
    [english_gold_by_word[word] for word in english_tuning_words],
    ignore_index=True,
)

english_heldout_gold = pd.concat(
    [english_gold_by_word[word] for word in english_heldout_words],
    ignore_index=True,
)

english_tuning_gold_path = corpus_gold_folder / "tuning_gold.jsonl"
english_heldout_gold_path = corpus_gold_folder / "heldout_gold.jsonl"

english_tuning_gold.to_json(
    english_tuning_gold_path,
    orient="records",
    lines=True,
)

english_heldout_gold.to_json(
    english_heldout_gold_path,
    orient="records",
    lines=True,
)

print("Tuning words:", english_tuning_gold["word"].nunique())
print("Heldout words:", english_heldout_gold["word"].nunique())

Tuning words: 1640
Heldout words: 704


In [85]:
english_tuning_distances_by_word = {}

for word in tqdm(
    english_tuning_words,
    desc="Preparing distances for cross-validation"
):
    english_tuning_distances_by_word[word] = get_word_distances(
        english_roberta_vectors_by_word[word],
        english_lexeme_vectors_by_word[word],
    )

Preparing distances for cross-validation: 100%|██████████| 1640/1640 [00:03<00:00, 453.03it/s]


In [86]:
def get_english_distance_scale(training_words):
    lexeme_pairs = []
    roberta_pairs = []

    for word in training_words:
        distances = english_tuning_distances_by_word[word]
        upper_triangle = np.triu_indices(
            len(distances["lexeme"]), k=1
        )

        lexeme_pairs.append(distances["lexeme"][upper_triangle])
        roberta_pairs.append(distances["roberta"][upper_triangle])

    lexeme_typical = np.median(np.concatenate(lexeme_pairs))
    roberta_typical = np.median(np.concatenate(roberta_pairs))

    if not np.isfinite(roberta_typical) or roberta_typical <= 0:
        raise ValueError("Cannot scale invalid or zero median distances.")

    return float(lexeme_typical / roberta_typical)

#### 5-Fold Cross Validation for Params

In [87]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_data = english_tuning.reset_index(drop=True)
cv_strata = cv_data["sense_count"].clip(upper=18)

In [105]:
def select_cv_parameters(summary, parameter_columns):
    f1_winners = summary["1a_F1"].eq(summary["1a_F1"].max())
    spearman_winners = summary["1b_Spearman"].eq(
        summary["1b_Spearman"].max()
    )

    shared_winner = summary.loc[f1_winners & spearman_winners]

    if len(shared_winner) == 1:
        selected = shared_winner.iloc[0][parameter_columns].to_dict()
        print("Selected parameters:", selected)
        return selected

    display(summary)
    return None

In [90]:
from evaluation import (
    load_subtask1_labels, compute_bcubed, calculate_JSD_distances,
    filter_jsd_by_min_instances, compute_spearman,
)

def evaluate_english_subtask1(gold_path, prediction_path):
    predicted_counts, gold_counts, predicted_labels, gold_labels = (
        load_subtask1_labels(prediction_path, gold_path)
    )

    # F1 includes every word.
    clustering = compute_bcubed(
        gold_labels, predicted_labels, prediction_path
    )

    # Temporal change requires at least two periods.
    temporal_gold_counts = {
        word: periods
        for word, periods in gold_counts.items()
        if len(periods) >= 2
    }
    temporal_predicted_counts = {
        word: predicted_counts[word]
        for word in temporal_gold_counts
    }

    gold_jsd = calculate_JSD_distances(temporal_gold_counts)
    predicted_jsd = calculate_JSD_distances(temporal_predicted_counts)
    gold_jsd = filter_jsd_by_min_instances(
        gold_jsd, temporal_gold_counts, min_instances=10
    )

    change = compute_spearman(
        gold_jsd, predicted_jsd, prediction_path
    )["rho"]

    return clustering, change

#### Lexeme

In [91]:
english_lexeme_cv_results = []
cv_validation_gold_path = corpus_gold_folder / "cv_gold.jsonl"
cv_validation_prediction_path = (
    PROJECT_ROOT / "outputs" / "predictions" / "ENG"
    / "cv_predictions.jsonl"
)

for fold, (_, cv_validation_word_indices) in enumerate(
    cv.split(cv_data, cv_strata), start=1
):
    cv_validation_words = cv_data.iloc[cv_validation_word_indices]["word"]

    cv_validation_gold = pd.concat(
        [english_gold_by_word[word] for word in cv_validation_words],
        ignore_index=True,
    )
    cv_validation_gold.to_json(cv_validation_gold_path, orient="records", lines=True)

    for cutoff in tqdm(
        np.arange(0.1, 1.01, 0.1), desc=f"XL-LEXEME fold {fold}"
    ):
        prediction_parts = []
        cluster_counts = []

        for word in cv_validation_words:
            vectors = normalize(english_lexeme_vectors_by_word[word])

            labels = AgglomerativeClustering(
                n_clusters=None,
                metric="cosine",
                linkage="average",
                distance_threshold=float(cutoff),
            ).fit_predict(vectors)

            cluster_counts.append(len(np.unique(labels)))

            predictions = english_data_by_word[word][
                ["word", "period_label", "sentence_id"]
            ].copy()
            predictions["label"] = [[int(label)] for label in labels]
            prediction_parts.append(predictions)

        predictions = pd.concat(prediction_parts, ignore_index=True)
        predictions.to_json(
            cv_validation_prediction_path, orient="records", lines=True
        )

        clustering, change = evaluate_english_subtask1(
            cv_validation_gold_path, cv_validation_prediction_path
        )

        english_lexeme_cv_results.append({
            "fold": fold,
            "distance_cutoff": round(float(cutoff), 2),
            "1a_F1": clustering["fcubed"],
            "1b_Spearman": change,
            "mean_clusters": np.mean(cluster_counts),
        })

english_lexeme_cv_results = pd.DataFrame(english_lexeme_cv_results)

english_lexeme_cv_summary = english_lexeme_cv_results.groupby(
    "distance_cutoff", as_index=False
)[["1a_F1", "1b_Spearman", "mean_clusters"]].mean()

english_lexeme_cv_summary

XL-LEXEME fold 1:  80%|████████  | 8/10 [03:15<00:51, 25.62s/it]/Users/kmonroygill/Desktop/semantic-change-over-time/src/evaluation/semeval2027-task3/evaluation.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pvalue = spearmanr(gold_scores, predicted_scores)
XL-LEXEME fold 1:  90%|█████████ | 9/10 [03:41<00:25, 25.82s/it]/Users/kmonroygill/Desktop/semantic-change-over-time/src/evaluation/semeval2027-task3/evaluation.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pvalue = spearmanr(gold_scores, predicted_scores)
XL-LEXEME fold 2:  80%|████████  | 8/10 [03:07<00:50, 25.10s/it]/Users/kmonroygill/Desktop/semantic-change-over-time/src/evaluation/semeval2027-task3/evaluation.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pvalue = spearmanr(gold_scores, predicted_scores)
XL-LEXEME fold 2:  90%|█████████ | 9/10 [03

,distance_cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.1,0.279872,0.248690,84.593293
1,0.2,0.496477,0.314002,29.707927
2,0.3,0.607358,0.334160,12.631707
3,0.4,0.642782,0.333929,5.825000
4,0.5,0.636022,0.288778,2.907317
5,0.6,0.602999,0.181314,1.550000
6,0.7,0.584821,0.066562,1.087195
7,0.8,0.582367,0.019208,1.009756
8,0.9,0.582017,0.000000,1.000000
9,1.0,0.582017,0.000000,1.000000


In [106]:
english_lexeme_selected_params = select_cv_parameters(
    english_lexeme_cv_summary,
    ["distance_cutoff"],
)

,distance_cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.1,0.279872,0.248690,84.593293
1,0.2,0.496477,0.314002,29.707927
2,0.3,0.607358,0.334160,12.631707
3,0.4,0.642782,0.333929,5.825000
4,0.5,0.636022,0.288778,2.907317
5,0.6,0.602999,0.181314,1.550000
6,0.7,0.584821,0.066562,1.087195
7,0.8,0.582367,0.019208,1.009756
8,0.9,0.582017,0.000000,1.000000
9,1.0,0.582017,0.000000,1.000000


In [107]:
if english_lexeme_selected_params is None:
    english_lexeme_selected_params = {
        "distance_cutoff": float(
            input("Enter the selected XL-LEXEME distance cutoff: ")
        )
    }

prediction_parts = []

for word in tqdm(english_heldout_words, desc="XL-LEXEME heldout"):
    vectors = normalize(english_lexeme_vectors_by_word[word])

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=english_lexeme_selected_params["distance_cutoff"],
    ).fit_predict(vectors)

    predictions = english_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    prediction_parts.append(predictions)

english_lexeme_heldout_predictions = pd.concat(
    prediction_parts, ignore_index=True
)
english_lexeme_heldout_predictions.to_json(
    english_heldout_prediction_folder / "lexeme_heldout_predictions.jsonl",
    orient="records",
    lines=True,
)

english_lexeme_heldout_scores, english_lexeme_heldout_change = (
    evaluate_english_subtask1(
        english_heldout_gold_path,
        english_heldout_prediction_folder / "lexeme_heldout_predictions.jsonl",
    )
)

print("XL-LEXEME heldout 1a F1:", english_lexeme_heldout_scores["fcubed"])
print("XL-LEXEME heldout 1b Spearman:", english_lexeme_heldout_change)

XL-LEXEME heldout: 100%|██████████| 704/704 [00:26<00:00, 26.26it/s]


XL-LEXEME heldout 1a F1: 0.6456136441406277
XL-LEXEME heldout 1b Spearman: 0.3085588719615503


#### Roberta

In [99]:
english_roberta_cv_results = []
cv_validation_gold_path = corpus_gold_folder / "cv_gold.jsonl"
cv_validation_prediction_path = (
    PROJECT_ROOT / "outputs" / "predictions" / "ENG"
    / "cv_predictions.jsonl"
)

for fold, (_, cv_validation_word_indices) in enumerate(
    cv.split(cv_data, cv_strata), start=1
):
    cv_validation_words = cv_data.iloc[cv_validation_word_indices]["word"]

    cv_validation_gold = pd.concat(
        [english_gold_by_word[word] for word in cv_validation_words],
        ignore_index=True,
    )
    cv_validation_gold.to_json(cv_validation_gold_path, orient="records", lines=True)

    for cutoff in tqdm(
        [
            0.002, 0.004, 0.006, 0.008, 0.010,
            0.012, 0.015, 0.020, 0.025, 0.030,
        ],
        desc=f"RoBERTa fold {fold}"
    ):
        prediction_parts = []
        cluster_counts = []

        for word in cv_validation_words:
            vectors = normalize(english_roberta_vectors_by_word[word])

            labels = AgglomerativeClustering(
                n_clusters=None,
                metric="cosine",
                linkage="average",
                distance_threshold=cutoff,
            ).fit_predict(vectors)

            cluster_counts.append(len(np.unique(labels)))

            predictions = english_data_by_word[word][
                ["word", "period_label", "sentence_id"]
            ].copy()
            predictions["label"] = [[int(label)] for label in labels]
            prediction_parts.append(predictions)

        predictions = pd.concat(prediction_parts, ignore_index=True)
        predictions.to_json(
            cv_validation_prediction_path, orient="records", lines=True
        )

        clustering, change = evaluate_english_subtask1(
            cv_validation_gold_path, cv_validation_prediction_path
        )

        english_roberta_cv_results.append({
            "fold": fold,
            "distance_cutoff": round(float(cutoff), 3),
            "1a_F1": clustering["fcubed"],
            "1b_Spearman": change,
            "mean_clusters": np.mean(cluster_counts),
        })

english_roberta_cv_results = pd.DataFrame(english_roberta_cv_results)

english_roberta_cv_summary = english_roberta_cv_results.groupby(
    "distance_cutoff", as_index=False
)[["1a_F1", "1b_Spearman", "mean_clusters"]].mean()

english_roberta_cv_summary

RoBERTa fold 5: 100%|██████████| 10/10 [03:32<00:00, 21.24s/it]


,distance_cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.002,0.081059,0.187962,184.656098
1,0.004,0.334586,0.213578,67.559756
2,0.006,0.472943,0.207802,31.440854
3,0.008,0.530556,0.202187,16.990854
4,0.010,0.556134,0.202837,9.975610
5,0.012,0.572603,0.199463,6.146341
6,0.015,0.581370,0.176001,3.229878
7,0.020,0.581503,0.126957,1.604268
8,0.025,0.582243,0.074214,1.196951
9,0.030,0.582123,0.036164,1.075610


In [108]:
english_roberta_selected_params = select_cv_parameters(
    english_roberta_cv_summary,
    ["distance_cutoff"],
)

,distance_cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.002,0.081059,0.187962,184.656098
1,0.004,0.334586,0.213578,67.559756
2,0.006,0.472943,0.207802,31.440854
3,0.008,0.530556,0.202187,16.990854
4,0.010,0.556134,0.202837,9.975610
5,0.012,0.572603,0.199463,6.146341
6,0.015,0.581370,0.176001,3.229878
7,0.020,0.581503,0.126957,1.604268
8,0.025,0.582243,0.074214,1.196951
9,0.030,0.582123,0.036164,1.075610


In [109]:
if english_roberta_selected_params is None:
    english_roberta_selected_params = {
        "distance_cutoff": float(
            input("Enter the selected XL-LEXEME distance cutoff: ")
        )
    }

prediction_parts = []

for word in tqdm(english_heldout_words, desc="RoBERTa heldout"):
    vectors = normalize(english_roberta_vectors_by_word[word])

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=english_roberta_selected_params["distance_cutoff"],
    ).fit_predict(vectors)

    predictions = english_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    prediction_parts.append(predictions)

english_roberta_heldout_predictions = pd.concat(
    prediction_parts, ignore_index=True
)
english_roberta_heldout_predictions.to_json(
    english_heldout_prediction_folder / "roberta_heldout_predictions.jsonl",
    orient="records",
    lines=True,
)

english_roberta_heldout_scores, english_roberta_heldout_change = (
    evaluate_english_subtask1(
        english_heldout_gold_path,
        english_heldout_prediction_folder / "roberta_heldout_predictions.jsonl",
    )
)

print("RoBERTa heldout 1a F1:", english_roberta_heldout_scores["fcubed"])
print("RoBERTa heldout 1b Spearman:", english_roberta_heldout_change)

RoBERTa heldout: 100%|██████████| 704/704 [00:18<00:00, 37.89it/s]


RoBERTa heldout 1a F1: 0.5754600415256017
RoBERTa heldout 1b Spearman: 0.18305943016786688


#### Blend

In [110]:
english_blend_cv_results = []

for fold, (cv_calibration_word_indices, cv_validation_word_indices) in enumerate(
    cv.split(cv_data, cv_strata), start=1
):
    cv_calibration_words = cv_data.iloc[cv_calibration_word_indices]["word"]
    cv_validation_words = cv_data.iloc[cv_validation_word_indices]["word"]

    cv_calibration_distance_scale = get_english_distance_scale(
        tuple(cv_calibration_words)
    )

    cv_validation_gold = pd.concat(
        [english_gold_by_word[word] for word in cv_validation_words],
        ignore_index=True,
    )
    cv_validation_gold.to_json(
        cv_validation_gold_path, orient="records", lines=True
    )

    for lexeme_weight in tqdm(
        [0.0, 0.25, 0.50, 0.75, 1.0],
        desc=f"Blend fold {fold}"
    ):
        for cutoff in [0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
            prediction_parts = []
            cluster_counts = []

            for word in cv_validation_words:
                word_distances = english_tuning_distances_by_word[word]

                distances = (
                    lexeme_weight * word_distances["lexeme"]
                    + (1 - lexeme_weight)
                    * cv_calibration_distance_scale
                    * word_distances["roberta"]
                )

                distances = np.maximum(distances, 0)
                distances = (distances + distances.T) / 2
                np.fill_diagonal(distances, 0)

                labels = AgglomerativeClustering(
                    n_clusters=None,
                    metric="precomputed",
                    linkage="average",
                    distance_threshold=cutoff,
                ).fit_predict(distances)

                cluster_counts.append(len(np.unique(labels)))

                predictions = english_data_by_word[word][
                    ["word", "period_label", "sentence_id"]
                ].copy()
                predictions["label"] = [[int(label)] for label in labels]
                prediction_parts.append(predictions)

            predictions = pd.concat(prediction_parts, ignore_index=True)
            predictions.to_json(
                cv_validation_prediction_path,
                orient="records",
                lines=True,
            )

            clustering, change = evaluate_english_subtask1(
                cv_validation_gold_path,
                cv_validation_prediction_path,
            )

            english_blend_cv_results.append({
                "fold": fold,
                "lexeme_weight": lexeme_weight,
                "cutoff": cutoff,
                "1a_F1": clustering["fcubed"],
                "1b_Spearman": change,
                "mean_clusters": np.mean(cluster_counts),
            })

english_blend_cv_results = pd.DataFrame(english_blend_cv_results)

english_blend_cv_summary = english_blend_cv_results.groupby(
    ["lexeme_weight", "cutoff"], as_index=False
)[["1a_F1", "1b_Spearman", "mean_clusters"]].mean()

english_blend_cv_summary

Blend fold 5: 100%|██████████| 5/5 [05:20<00:00, 64.14s/it]


,lexeme_weight,cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.00,0.10,0.080679,0.187843,185.081098
1,0.00,0.15,0.210343,0.204436,109.777439
2,0.00,0.20,0.333774,0.213651,67.891463
3,0.00,0.30,0.472374,0.207181,31.614634
4,0.00,0.40,0.530103,0.201250,17.107927
5,0.00,0.50,0.555974,0.202282,10.056707
6,0.25,0.10,0.080189,0.204111,193.971951
7,0.25,0.15,0.202650,0.235710,119.108537
8,0.25,0.20,0.337507,0.261849,73.234756
9,0.25,0.30,0.508651,0.266983,30.953049


In [111]:
english_blend_selected_params = select_cv_parameters(
    english_blend_cv_summary,
    ["lexeme_weight", "cutoff"],
)

,lexeme_weight,cutoff,1a_F1,1b_Spearman,mean_clusters
0,0.00,0.10,0.080679,0.187843,185.081098
1,0.00,0.15,0.210343,0.204436,109.777439
2,0.00,0.20,0.333774,0.213651,67.891463
3,0.00,0.30,0.472374,0.207181,31.614634
4,0.00,0.40,0.530103,0.201250,17.107927
5,0.00,0.50,0.555974,0.202282,10.056707
6,0.25,0.10,0.080189,0.204111,193.971951
7,0.25,0.15,0.202650,0.235710,119.108537
8,0.25,0.20,0.337507,0.261849,73.234756
9,0.25,0.30,0.508651,0.266983,30.953049


In [112]:
if english_blend_selected_params is None:
    english_blend_selected_params = {
        "lexeme_weight": float(input("Enter selected Lexeme weight: ")),
        "cutoff": float(input("Enter selected blend cutoff: ")),
    }

english_settings = {
    "blend": {
        "cutoff": english_blend_selected_params["cutoff"],
        "lexeme_weight": english_blend_selected_params["lexeme_weight"],
        "distance_scale": get_english_distance_scale(
            tuple(english_tuning_words)
        ),
    },
}

pd.DataFrame(english_settings).T

,cutoff,lexeme_weight,distance_scale
blend,0.4,0.75,50.132526


In [113]:
if english_blend_selected_params is None:
    raise ValueError("Complete blend parameter review first.")

prediction_parts = []

for word in tqdm(english_heldout_words, desc="Blend heldout"):
    lexeme_matrix = pairwise_distances(
        normalize(english_lexeme_vectors_by_word[word]),
        metric="cosine",
    )
    roberta_matrix = pairwise_distances(
        normalize(english_roberta_vectors_by_word[word]),
        metric="cosine",
    )

    distances = (
        english_blend_selected_params["lexeme_weight"] * lexeme_matrix
        + (1 - english_blend_selected_params["lexeme_weight"])
        * english_settings["blend"]["distance_scale"]
        * roberta_matrix
    )

    distances = np.maximum(distances, 0)
    distances = (distances + distances.T) / 2
    np.fill_diagonal(distances, 0)

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=english_blend_selected_params["cutoff"],
    ).fit_predict(distances)

    predictions = english_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    prediction_parts.append(predictions)

english_blend_heldout_predictions = pd.concat(
    prediction_parts, ignore_index=True
)
english_blend_heldout_predictions.to_json(
    english_heldout_prediction_folder / "blend_heldout_predictions.jsonl",
    orient="records",
    lines=True,
)

english_blend_heldout_scores, english_blend_heldout_change = (
    evaluate_english_subtask1(
        english_heldout_gold_path,
        english_heldout_prediction_folder / "blend_heldout_predictions.jsonl",
    )
)

print("Blend heldout 1a F1:", english_blend_heldout_scores["fcubed"])
print("Blend heldout 1b Spearman:", english_blend_heldout_change)

Blend heldout: 100%|██████████| 704/704 [00:02<00:00, 246.87it/s]


Blend heldout 1a F1: 0.6447951888927806
Blend heldout 1b Spearman: 0.3251587866865655


In [115]:
english_best_settings = {
    "clustering": "AgglomerativeClustering",
    "linkage": "average",
    "labels_per_usage": 1,
    "tuning_words": english_tuning_words,
    "reserved_words": english_heldout_words,
    "approaches": {
        "lexeme": {
            "embedding_model": lexeme_name,
            "metric": "cosine",
            "distance_cutoff": float(
                english_lexeme_selected_params["distance_cutoff"]
            ),
        },
        "roberta": {
            "embedding_model": roberta_name,
            "metric": "cosine",
            "distance_cutoff": float(
                english_roberta_selected_params["distance_cutoff"]
            ),
        },
        "blend": {
            "embedding_models": [lexeme_name, roberta_name],
            "metric": "precomputed",
            "distance_cutoff": float(english_settings["blend"]["cutoff"]),
            "lexeme_weight": float(english_settings["blend"]["lexeme_weight"]),
            "distance_scale": float(english_settings["blend"]["distance_scale"]),
        },
    },
}

settings_folder = PROJECT_ROOT / "outputs" / "settings"
settings_folder.mkdir(parents=True, exist_ok=True)

(settings_folder / "subtask1_best_settings_ENG.json").write_text(
    json.dumps(english_best_settings, indent=2, ensure_ascii=False),
    encoding="utf-8",
);

In [116]:
english_heldout_results = pd.DataFrame(
    [
        ("XL-LEXEME", english_lexeme_heldout_scores["fcubed"], english_lexeme_heldout_change),
        ("XLM-RoBERTa", english_roberta_heldout_scores["fcubed"], english_roberta_heldout_change),
        ("Blend", english_blend_heldout_scores["fcubed"], english_blend_heldout_change),
    ],
    columns=["approach", "1a_F1", "1b_Spearman"],
)

english_evaluation_folder = (
    PROJECT_ROOT / "outputs" / "evaluations" / "ENG"
)
english_evaluation_folder.mkdir(parents=True, exist_ok=True)

english_heldout_results.to_csv(
    english_evaluation_folder / "subtask1_heldout_comparison.csv",
    index=False,
)

print(english_heldout_results.to_csv(index=False))

approach,1a_F1,1b_Spearman
XL-LEXEME,0.6456136441406277,0.3085588719615503
XLM-RoBERTa,0.5754600415256017,0.18305943016786688
Blend,0.6447951888927806,0.3251587866865655



### 3. Swedish - Using English Settings

#### Load Swedish Words

In [120]:
SV_data_by_word = {
    path.name.removesuffix("_usages.jsonl"): pd.read_json(path, lines=True)
    for path in sorted(cache_folder.glob("*_usages.jsonl"))
}

with np.load(cache_folder / "xl_lexeme_vectors.npz") as cache:
    swedish_lexeme_vectors.update({
        word: cache[word] for word in SV_data_by_word
    })

with np.load(cache_folder / "xlmr_vectors.npz") as cache:
    swedish_roberta_vectors.update({
        word: cache[word] for word in SV_data_by_word
    })

SV_gold_path = (
    PROJECT_ROOT / "data" / "processed" / "gold" / "SV" / "subtask1.jsonl"
)
SV_gold = pd.read_json(SV_gold_path, lines=True)

print("Loaded SV words:", sorted(SV_data_by_word))

Loaded SV words: ['dumpa', 'exportera', 'fru', 'fröken', 'förort', 'herre', 'kvinna', 'motiv', 'skär', 'suga']


#### Lexeme

In [121]:
SV_lexeme_prediction_parts = []

for word in tqdm(sorted(SV_data_by_word), desc="SV Lexeme"):
    vectors = normalize(swedish_lexeme_vectors[word])

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=english_best_settings["approaches"]["lexeme"]["distance_cutoff"],
    ).fit_predict(vectors)

    predictions = SV_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    SV_lexeme_prediction_parts.append(predictions)

SV_lexeme_predictions = pd.concat(
    SV_lexeme_prediction_parts, ignore_index=True
)

SV Lexeme: 100%|██████████| 10/10 [00:00<00:00, 45.61it/s]


In [122]:
SV_prediction_folder = PROJECT_ROOT / "outputs" / "predictions" / "SV" / "english_settings"
SV_prediction_folder.mkdir(parents=True, exist_ok=True)

SV_lexeme_prediction_path = (
    SV_prediction_folder / "lexeme.jsonl"
)

SV_lexeme_predictions.to_json(
    SV_lexeme_prediction_path,
    orient="records",
    lines=True,
)

In [123]:
SV_lexeme_scores, SV_lexeme_change = evaluate_subtask1(
    SV_gold_path,
    SV_lexeme_prediction_path,
)

print("approach,1a_F1,1b_Spearman")
print(
    f"XL-LEXEME,{SV_lexeme_scores['fcubed']},{SV_lexeme_change}"
)

approach,1a_F1,1b_Spearman
XL-LEXEME,0.8108291081721418,0.5620164713399316


#### Roberta

In [124]:
SV_roberta_prediction_parts = []

for word in tqdm(sorted(SV_data_by_word), desc="SV RoBERTa"):
    vectors = normalize(swedish_roberta_vectors[word])

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=english_best_settings["approaches"]["roberta"]["distance_cutoff"],
    ).fit_predict(vectors)

    predictions = SV_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    SV_roberta_prediction_parts.append(predictions)

SV_roberta_predictions = pd.concat(
    SV_roberta_prediction_parts, ignore_index=True
)

SV RoBERTa: 100%|██████████| 10/10 [00:00<00:00, 63.10it/s]


In [125]:
SV_roberta_prediction_path = SV_prediction_folder / "roberta.jsonl"

SV_roberta_predictions.to_json(
    SV_roberta_prediction_path,
    orient="records",
    lines=True,
)

In [126]:
SV_roberta_scores, SV_roberta_change = evaluate_subtask1(
    SV_gold_path,
    SV_roberta_prediction_path,
)

print("approach,1a_F1,1b_Spearman")
print(
    f"XLM-RoBERTa,{SV_roberta_scores['fcubed']},{SV_roberta_change}"
)

approach,1a_F1,1b_Spearman
XLM-RoBERTa,0.6907211589729247,0.5739899369593245


#### 75/25 Blend

In [127]:
SV_blend_prediction_parts = []

for word in tqdm(sorted(SV_data_by_word), desc="SV Blend"):
    lexeme_matrix = pairwise_distances(
        normalize(swedish_lexeme_vectors[word]),
        metric="cosine",
    )
    roberta_matrix = pairwise_distances(
        normalize(swedish_roberta_vectors[word]),
        metric="cosine",
    )

    distances = (
        english_best_settings["approaches"]["blend"]["lexeme_weight"]
        * lexeme_matrix
        + (1 - english_best_settings["approaches"]["blend"]["lexeme_weight"])
        * english_best_settings["approaches"]["blend"]["distance_scale"]
        * roberta_matrix
    )

    distances = np.maximum(distances, 0)
    distances = (distances + distances.T) / 2
    np.fill_diagonal(distances, 0)

    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=english_best_settings["approaches"]["blend"]["distance_cutoff"],
    ).fit_predict(distances)

    predictions = SV_data_by_word[word][
        ["word", "period_label", "sentence_id"]
    ].copy()
    predictions["label"] = [[int(label)] for label in labels]
    SV_blend_prediction_parts.append(predictions)

SV_blend_predictions = pd.concat(
    SV_blend_prediction_parts, ignore_index=True
)

SV Blend: 100%|██████████| 10/10 [00:00<00:00, 223.42it/s]


In [128]:
SV_blend_prediction_path = SV_prediction_folder / "blend.jsonl"

SV_blend_predictions.to_json(
    SV_blend_prediction_path,
    orient="records",
    lines=True,
)

In [129]:
SV_blend_scores, SV_blend_change = evaluate_subtask1(
    SV_gold_path,
    SV_blend_prediction_path,
)

print("approach,1a_F1,1b_Spearman")
print(
    f"Blend,{SV_blend_scores['fcubed']},{SV_blend_change}"
)

approach,1a_F1,1b_Spearman
Blend,0.7846541375504706,0.506744756136961


In [130]:
SV_english_settings_results = pd.DataFrame(
    [
        ("XL-LEXEME", SV_lexeme_scores["fcubed"], SV_lexeme_change),
        ("XLM-RoBERTa", SV_roberta_scores["fcubed"], SV_roberta_change),
        ("Blend", SV_blend_scores["fcubed"], SV_blend_change),
    ],
    columns=["approach", "1a_F1", "1b_Spearman"],
)

SV_evaluation_folder = (
    PROJECT_ROOT / "outputs" / "evaluations" / "SV"
)
SV_evaluation_folder.mkdir(parents=True, exist_ok=True)

SV_english_settings_results.to_csv(
    SV_evaluation_folder / "subtask1_SV_with_english_settings.csv",
    index=False,
)

print(SV_english_settings_results.to_csv(index=False))

approach,1a_F1,1b_Spearman
XL-LEXEME,0.8108291081721418,0.5620164713399316
XLM-RoBERTa,0.6907211589729247,0.5739899369593245
Blend,0.7846541375504706,0.506744756136961

